In [1]:
import sqlite3
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModel, AutoTokenizer

In [2]:
from huggingface_hub import notebook_login
notebook_login()

In [4]:
model_id = "FlagAlpha/Llama2-Chinese-7b-Chat"
offload_folder_path = "../offload_folder"


tokenizer = AutoTokenizer.from_pretrained(model_id,use_fast=False, return_tensors="pt")
model = transformers.AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map='auto',
    offload_folder = offload_folder_path
).half()

model.eval()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/Users/ender_yang/opt/anaconda3/envs/huggingface/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:492: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`. This was detected when initializing the generation config instance, which means the corresponding file may hold incorrect parameterization and should be fixed.
  warnings.warn(
/Users/ender_yang/opt/anaconda3/envs/huggingface/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:497: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`. This was detected when initializing the generation config instance, which means the corresponding file may hold incorrect parameterization and should be fixed.
  warnings.warn(
/U

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 4096, padding_idx=0)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (up_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm()
        (post_attention_layernorm): LlamaRMSNorm()
      )
    )
    (norm): LlamaRMSNorm()
 

In [7]:
prompt = "你好"

In [9]:
generator = transformers.pipeline(
        model=model, 
        tokenizer=tokenizer,
        task='text-generation',
        torch_dtype=torch.float16,
        device_map='auto',
        temperature=0.1,
        repetition_penalty=1.1,
        do_sample=True
    )
res = generator(prompt,num_return_sequences=1, eos_token_id = tokenizer.eos_token_id)
print(res[0]["generated_text"])

/Users/ender_yang/opt/anaconda3/envs/huggingface/lib/python3.9/site-packages/transformers/generation/utils.py:1197: UserWarning: You have modified the pretrained model configuration to control generation. This is a deprecated strategy to control generation and will be removed soon, in a future version. Please use and modify the model generation configuration (see https://huggingface.co/docs/transformers/generation_strategies#default-text-generation-configuration )
  warnings.warn(


KeyboardInterrupt: 

In [1]:
from typing import List

from langchain.chains import create_structured_output_runnable
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.pydantic_v1 import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_text_splitters import TokenTextSplitter

document_content = """
【法宝引证码】 CLI.12.1541468
原文链接：https://www.pkulaw.com/lar/32f09a1c6c282205e761c51231946224bdfb.html
广东省住房和城乡建设厅关于印发环卫行业信用管理暂行办法的通知
广东省住房和城乡建设厅关于印发环卫行业信用管理暂行办法的通知

（粤建规范〔2019〕4号）

各地级以上市环境卫生主管部门：

　　现将《广东省住房和城乡建设厅关于环卫行业信用管理暂行办法》印发给你们，请认真组织实施。实施过程中遇到的问题，请径向省住房城乡建设厅反映。
广东省住房和城乡建设厅

2019年10月11日
　　广东省住房和城乡建设厅关于环卫行业信用管理暂行办法
　第一章　总　则
　　第一条　（目的依据）

　　为了规范广东省环卫行业秩序，建立生活垃圾处理运营单位信用体系，促进环卫行业健康、有序发展，根据《企业信息公示暂行条例》《广东省城乡生活垃圾处理条例》等相关法律法规，结合广东省环卫行业实际，制定本办法。
　　第二条　（适用范围）

　　在广东省行政区域内从事生活垃圾清扫、分类、收集、运输、处置等环卫服务活动的企业纳入信用管理范围。
　　第三条　（管理内容）

　　广东省环卫行业信用管理包括企业信用信息、项目信用综合考评、企业重点监管名单三项内容。
　　第四条　（管理原则）

　　环卫信用信息登记遵循公开、公平、公正的原则，以信用体系建设为主导，以政府监督管理为手段，以信用信息登记为载体，实行环卫信用管理制度，强化主体自律和社会监督，激发市场活力。
　　第五条　（管理部门和管理系统）

　　省住房城乡建设主管部门负责指导全省生活垃圾处理运营单位信用体系建设工作，建立和完善广东省环卫行业信用管理平台（以下简称“信用管理平台”）。

　　各地级以上市环境卫生主管部门根据管理权限对辖区内环卫企业的行为及信用信息实施管理，负责信用信息监管工作。

　第二章　信用信息管理
　　第六条　（信用信息采集、审核、录入、变更及注销）

　　各地级以上市环境卫生主管部门负责企业信用信息的采集、审核和录入，并及时对信用管理平台的信用信息进行变更和注销。
　　第七条　（信用信息明细）

　　信用信息包括企业名称、统一社会信用代码、法定代表人、委托单位、项目名称、项目负责人、项目地点、项目总额、服务期限、《广东省城市生活垃圾经营性清扫、收集、运输服务许可证》或《广东省城市生活垃圾经营性处置服务许可证》证书编号、证书有效期、联系电话等。
　　第八条　（责任落实）

　　各地级以上市环境卫生主管部门对其录入信用管理平台的信息真实性、客观性、完整性负责。
　　第九条　（信息公布）

　　各地级以上市环境卫生主管部门按照公开、公平、公正的原则，及时将企业信用信息和项目信用综合考评结果在信用管理平台公布，供各地级以上市环境卫生主管部门查询使用。

　第三章　项目信用综合考评
　　第十条　（考评单位）

　　纳入信用管理范围的企业，由各地级以上市环境卫生主管部门统一进行年度项目信用综合考评。
　　第十一条　（考评内容）

　　项目信用综合考评以企业履约能力、服务质量、遵纪守法三项指标作为考评内容。
　　第十二条　（考评等级）

　　项目信用综合考评等级分为优、良、差三个等级。
　　第十三条　（考评时间）

　　各地级以上市环境卫生主管部门在每年的1月完成上一年度的项目信用综合考评。

　第四章　监督管理
　　第十四条　（分级分类监管）

　　各地级以上市环境卫生主管部门根据企业的年度项目信用综合考评等级，实施分级分类监管，建立企业重点监管名单制度。

　　（一）全部项目信用综合考评等级为优的企业，纳入监管对象，实行动态监督管理，优先推荐参加政府招投标项目、评优评先活动；

　　（二）全部项目信用综合考评等级为良的企业，纳入监管的主要对象，实行常态化监督管理；

　　（三）项目信用综合考评为差的企业，列入重点监管名单，纳入重点监管对象，实行严格的日常监督检查；
　　第十五条　（重点监管名单认定）

　　企业在当年度有下列情况之一的，列入企业重点监管名单，期限为一年，主要包括：

　　（一）上一年度项目信用综合考评为差的；

　　（二）在环卫招标投标活动中存在违法行为且受到行政处罚的；

　　（三）发生较大及以上生产安全责任事故，或1年内累计发生2次及以上一般生产安全责任事故的； 

　　（四）违法运输、倾倒或处置生活垃圾，受到行政处罚的；

　　（五）拖欠工人工资拒不整改的。
　　第十六条　（认定录入）

　　各地级以上市环境卫生主管部门认定企业重点监管名单后，应在5个工作日内书面通知企业，并在10个工作日内登陆信用管理平台录入相关信息。
　　第十七条　（重点监管名单的移出）

　　列入重点监管名单的企业，自录入信用管理平台之日起一年内未再发生本办法第十五条规定的情形的，由项目所在地地级以上市环境卫生主管部门核准后移出企业重点监管名单。
　　第十八条　（异议处理）

　　企业对被列入重点监管名单有异议的，应当在收到书面通知之日起5个工作日内，向作出认定的部门提出书面复核申请。
　　第十九条　（复核处理）

　　各地级以上市环境卫生主管部门在收到企业提出的书面复核申请后5个工作日内进行复核并书面答复。

　第五章　附则
　　第二十条　（解释单位）

　　本办法由广东省住房和城乡建设厅负责解释。
　　第二十一条　（试行日期）

　　本办法自2019年10月10日起施行，有效期为3年。

　　广东省环卫信用信息登记表
企业名称
统一社会
信用代码
法定代表人
项目名称
委托单位
项目地点
项目总额
服务期限
许可证编号
许可证有效期
项目负责人
联系电话
项目委托单位意见
项目委托单位（盖章）：
年   月   日
地级以上市环境卫生主管
部门意见
地级以上市环境卫生主管部门（盖章）：
年   月   日

　　注：各地级以上市环境卫生主管部门负责采集、审核企业信用信息，并录入信用管理平台。

　　广东省环卫项目信用综合考评表
企业名称
项目名称
统一社会
信用代码
项目负责人
考评年度
联系电话
履约能力
是否存在违约行为        是□  否□
是否按期履行合同        是□  否□
综合履约能力            优□  良□  差□
服务质量
是否建立规章制度        是□  否□
是否发生安全生产事故    是□  否□
业主满意度              优□  良□  差□
遵纪守法
是否按规定为员工购买社保              是□  否□
是否按规定支付工人工资                是□  否□
是否受到各级环境卫生主管部门行政处罚  是□  否□
项目委托单位意见
项目委托单位（盖章）：
年   月   日
地级以上市环境卫生主管
部门意见
地级以上市环境卫生主管部门（盖章）：
年   月   日

　　注：（1）综合履约能力、业主满意度为“优”，且遵纪守法的企业考评等级为“优”。

　　（2）综合履约能力、业主满意度为“良”及以上但未达到全“优”，且遵纪守法的企业考评等级为“良”。

　　（3）综合履约能力、业主满意度为“差”，或发生违法违规行为的企业考评等级为“差”。

　　（4）发生违法违规行为的企业考评等级为“差”。

　　广东省环卫企业重点监管名单确认表
企业名称
项目名称
统一社会
信用代码
法定代表人
项目负责人
联系电话
1
上一年度项目信用综合考评为差的。
有□  无□
2
在环卫招标投标活动中存在违法行为且受到行政处罚的。
有□  无□
3
发生较大及以上生产安全责任事故，或1年内累计发生2次及以上一般生产安全责任事故的。
有□  无□
4
违法运输、倾倒或处置生活垃圾，受到行政处罚的。
有□  无□
5
拖欠工人工资拒不整改的。
有□  无□
项目委托单位意见
项目委托单位（盖章）：
年   月   日
地级以上市环境卫生主管
部门意见
地级以上市环境卫生主管部门（盖章）：
年   月   日

　　注：企业发生上述情况中的任意一项，经各地级以上市环境卫生主管部门确认即列入企业重点监管名单，期限为一年。

　　《广东省城市生活垃圾经营性清扫、收集、运输服务许可证》统计情况表
序号
法人名称
办公住所
设施地址
核准经营范围
核准经营车辆 
法定代表人
联系人
联系电话
许可证有效期
证书编号
申请日期
备注

　　《广东省城市生活垃圾经营性处置服务许可证》统计情况表
序号
法人名称
办公住所
设施地址
核准经营范围
核准经营规模（吨/年）
法定代表人
联系人
联系电话
许可证有效期
证书编号
申请日期
备注

　　餐厨垃圾收运企业情况表
地市
序号
企业名称
企业负责人
联系电话
业务范围
许可证
餐厨垃圾种类（餐饮垃圾、居民厨余垃圾、农贸市场有机易腐垃圾）
收集频次
收运方式
收运车辆数量与类型
收运
范围
收运
线路
备注
（次/天）
（直收直运/中转转运）
"""

class PolicyContent(BaseModel):
    """Details of the extracted policy content."""
    content: str = Field(
        ..., description="包含一般政策内容的段落。"
    )
    reason: str = Field(
        ..., description="请解释为什么这些段落包含了一般政策内容。"
    )

class ExtractionData(BaseModel):
    """Extracted information representing the general policy content."""
    policy_contents: List[PolicyContent]

system_prompt = """
你将要阅读一段详细的政策文本。你的任务是找到描述一般政策内容的部分。为了确保找到的内容是最相关的，你需要遵循一系列详细的步骤和条件。

步骤1：识别启动词，前置段落
首先，识别文本中包含以下启动词的段落："现提出…", "提出如下…", "提出以下…", "现将…", "现就…", "现…如下"。这些段落通常会在政策内容之前出现。

步骤2：检查实施依据
对于每个包含启动词的段落，检查是否包含以下实施依据："为进一步贯彻落实", "根据", "按照xx要求，依据xx规定"。这些表达帮助确认段落的政策性质。

步骤3：补充条件
a. 检查这些段落是否提到国家最高领导人或机构（如"胡", "温", "习", "李", "党中央", "国务院"）。
b. 如果文本中有"总则"，包含该关键词的第一条段落可能也是重要的。
c. 查找包含"指导思想"或"基本原则"关键词的段落，这些通常描述政策的基本方针。
d. 如果以上条件未满足，寻找执行单位名称之后的第一个段落。

步骤4：汇总与评估
将所有符合条件的段落汇总，根据其重要性和相关性进行评估，确定哪些段落最能代表"一般政策内容"。

##例子：
"为贯彻落实全市安全生产工作会议精神及中央、省、市关于安全生产的决策部署，充分发挥财政职能作用，切实加强当前安全生产工作，现结合财政工作实际，制定本制度。
　　一、总体要求
　　财政安全生产工作的总体要求是：以习近平总书记、李克强总理关于安全生产的重要指示批示精神为指导，以贯彻落实党中央、国务院《关于推进安全生产领域改革发展的意见》为主线，以杜绝安全事故为目标，坚持标本兼治，强化综合治理，加强风险管控，全面推进财政安全生产领域改革创新，推动财政安全生产工作水平全面提升。
"""

prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{text}"),
    ]
)

llm = ChatOpenAI(
    model="gpt-4-0125-preview",
    temperature=0,  # Important for extraction tasks
)

extractor = prompt | llm.with_structured_output(
    schema=ExtractionData,
    method="function_calling",
    include_raw=False,
)



text_splitter = TokenTextSplitter(
    chunk_size=2000,  # Adjust based on your LLM's limitations
    chunk_overlap=20,  # Some overlap to ensure nothing gets cut off mid-context
)

# Assuming `document_content` is your loaded and prepared policy text
texts = text_splitter.split_text(document_content)

# Extract from the first few chunks as an example
first_few = texts[:3]  # Adjust based on your needs

extractions = extractor.batch(
    [{"text": text} for text in first_few],
    {"max_concurrency": 5},  # Adjust concurrency based on your setup
)

policy_contents = []

for extraction in extractions:
    policy_contents.extend(extraction.policy_contents)

# Now `policy_contents` holds all extracted general policy content sections


ModuleNotFoundError: No module named 'langchain'